# PID-RL API Server for HackathonModelo: Fenryr/qwen2.5-1.5B-pid-v1 (LoRA adapter)## Quick Start1. Runtime -> Change runtime type -> GPU (T4)2. Run cells in order3. Enter your HF token when prompted4. Use the public URL for your app

## 1 - Install Dependencies

In [ ]:
!pip install --upgrade pip -q!pip install transformers>=4.51.3 unsloth peft accelerate -q!pip install fastapi uvicorn pydantic cloudflared -q!pip install yfinance -qprint('Dependencies installed!')

## 2 - Enter HF Token

In [ ]:
from getpass import getpassHF_TOKEN = getpass('Enter HF token: ')import osos.environ['HF_TOKEN'] = HF_TOKENprint('Token set!')

## 3 - Clone Repo

In [ ]:
import subprocess, sys, osREPO_DIR = '/content/qwen-rl-shanghai'if not os.path.isdir(REPO_DIR):    subprocess.run(['git', 'clone', 'https://github.com/Grinta-Protocol/qwen-rl-shanghai.git', REPO_DIR], check=True)sys.path.insert(0, f'{REPO_DIR}/pid_rl')os.chdir(f'{REPO_DIR}/pid_rl')print('Repo cloned!')

## 4 - Load Model v1

In [ ]:
from unsloth import FastLanguageModelimport torchfrom peft import PeftModelLORA_PATH = 'Fenryr/qwen2.5-1.5B-pid-v1'BASE_MODEL = 'unsloth/Qwen2.5-1.5B-Instruct'MAX_SEQ_LEN = 2048print('Loading base model...')model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=False, fast_inference=False)print('Loading LoRA adapter...')model = FastLanguageModel.get_peft_model(model, r=8, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_alpha=8)model = PeftModel.from_pretrained(model, LORA_PATH)FastLanguageModel.for_inference(model)print('Model loaded!')

## 5 - Inference Wrapper

In [ ]:
from prompt import build_prompt, parse_outputfrom types import SimpleNamespaceimport numpy as npclass TrainedModelPolicy:    name = 'trained'    def __init__(self, model, tokenizer, max_new_tokens=256, temperature=0.3):        self.model = model        self.tokenizer = tokenizer        self.max_new_tokens = max_new_tokens        self.temperature = temperature    def decide(self, state, btc_history, deviation_history):        sys_p, user_p = build_prompt(state=state, btc_history=btc_history, deviation_history=deviation_history)        messages = [{'role': 'system', 'content': sys_p}, {'role': 'user', 'content': user_p}]        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)        inputs = self.tokenizer(text, return_tensors='pt').to(self.model.device)        with torch.no_grad():            out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, temperature=self.temperature, do_sample=True, pad_token_id=self.tokenizer.eos_token_id)        gen = self.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)        return gentrained_policy = TrainedModelPolicy(model, tokenizer)print('Inference wrapper ready!')

## 6 - FastAPI Server

In [ ]:
from fastapi import FastAPI, HTTPExceptionfrom pydantic import BaseModelimport threading, uvicornapp = FastAPI(title='PID-RL v1 API')class InferenceRequest(BaseModel):    state: dict    btc_history: list    deviation_history: listclass InferenceResponse(BaseModel):    kp: float    ki: float    raw_output: str    success: bool@app.get('/')def root():    return {'status': 'ok', 'model': 'Fenryr/qwen2.5-1.5B-pid-v1'}@app.get('/health')def health():    return {'status': 'healthy', 'model': 'v1-lora'}@app.post('/predict', response_model=InferenceResponse)def predict(req: InferenceRequest):    try:        state = SimpleNamespace(**req.state)        output = trained_policy.decide(state=state, btc_history=req.btc_history, deviation_history=req.deviation_history)        try:            parsed = parse_output(output)        except:            parsed = {}        return InferenceResponse(kp=parsed.get('kp', state.kp), ki=parsed.get('ki', state.ki), raw_output=output[:500], success=True)    except Exception as e:        raise HTTPException(status_code=500, detail=str(e))def run_server():    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')threading.Thread(target=run_server, daemon=True).start()import time; time.sleep(2)print('API Server ready on port 8000')

## 7 - Cloudflare Tunnel

In [ ]:
!pip install cloudflared -qprint('Cloudflared installed')

In [ ]:
import subprocess, timesubprocess.Popen(['nohup', 'cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)time.sleep(10)log = open('/tmp/tunnel.log').read()if 'trycloudflare.com' in log:    for line in log.splitlines():        if 'https://' in line and 'trycloudflare.com' in line:            url = line.strip().split()[0]            break    print('='*60)    print('TUNNEL READY!')    print('='*60)    print('Public URL:', url)    print('POST', url + '/predict')    print('Keep cell running!')else:    print('Waiting...')    print(log)

## 8 - Test (Optional)

In [ ]:
import requests, jsontest = {'state': {'deviation_pct': 2.5, 'kp': 1.0, 'ki': 0.01, 'integral_error': 5.0}, 'btc_history': [42000, 42100, 41900, 41800, 41750], 'deviation_history': [1.2, 1.5, 1.8, 2.0, 2.2]}try:    r = requests.post('http://localhost:8000/predict', json=test, timeout=60)    print('OK!', json.dumps(r.json(), indent=2))except Exception as e:    print('Fail:', e)

---Note: v2 corrupted, using v1Tips: Run cells in order